In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import os



In [2]:
# إنشاء مجلد للرسومات لو مش موجود
os.makedirs('visuals', exist_ok=True)

# قراءة الملفات (عدّل الأسماء لو مختلفة عندك
fact_sales = pd.read_csv("fact_sales.csv")
dim_products = pd.read_csv("dim_products.csv")
dim_customers = pd.read_csv("dim_customers.csv")

# دمج الجداول
df = fact_sales.merge(dim_customers, on='CustomerID').merge(dim_products, on='StockCode')
df['Revenue'] = df['Quantity'] * df['UnitPrice']

In [3]:
df.is_cancelled.unique()

array([False,  True])

In [4]:
# ============================================
# 1. Top 10 Countries by Revenue
# ============================================
top_countries = df[df['is_cancelled'] == False].groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10,6))
top_countries.plot(kind='bar', color='steelblue')
plt.xlabel('Revenue')
plt.title('Top 10 Countries by Revenue')
plt.tight_layout()
plt.savefig('visuals/top_countries.png')
plt.close()

In [5]:
# ============================================
# 2. Sales Trend by Month
# ============================================
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M').astype(str)
monthly_sales = df[df['is_cancelled'] == False].groupby('YearMonth')['Revenue'].sum()

plt.figure(figsize=(12,6))
monthly_sales.plot(kind='line', marker='o', color='darkgreen')
plt.xlabel('Month')
plt.ylabel('Revenue')
plt.title('Monthly Sales Trend')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('visuals/monthly_trend.png')
plt.close()



In [6]:
# ============================================
# 3. Cancellation Rate by Country (Top 10 by order count)
# ============================================
country_stats = df.groupby('Country').agg(
    total_orders=('InvoiceNo', 'nunique')
)
cancelled_by_country = df[df['is_cancelled'] == True].groupby('Country')['InvoiceNo'].nunique()
country_stats['cancelled_orders'] = cancelled_by_country
country_stats['cancelled_orders'] = country_stats['cancelled_orders'].fillna(0)
country_stats['cancellation_rate'] = (country_stats['cancelled_orders'] / country_stats['total_orders'] * 100).round(2)
country_stats['cancellation_rate'] = (country_stats['cancelled_orders'] / country_stats['total_orders'] * 100).round(2)
top_by_volume = country_stats[country_stats['total_orders'] >= 20].sort_values('cancellation_rate', ascending=False).head(10)

plt.figure(figsize=(10,6))
top_by_volume['cancellation_rate'].plot(kind='barh', color='indianred')
plt.xlabel('Cancellation Rate (%)')
plt.title('Top 10 Countries by Cancellation Rate')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('visuals/cancellation_by_country.png')
plt.close()


In [7]:
# ============================================
# 4. Order Value: Cancelled vs Not Cancelled
# ============================================
avg_price = df.groupby('is_cancelled')['UnitPrice'].mean()

plt.figure(figsize=(6,6))
avg_price.plot(kind='bar', color=['seagreen', 'indianred'])
plt.ylabel('Average Unit Price')
plt.title('Average Product Price: Cancelled vs Not Cancelled')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('visuals/price_vs_cancellation.png')
plt.close()

print("تم حفظ 4 رسومات بمجلد visuals/")

تم حفظ 4 رسومات بمجلد visuals/
